In [ ]:
"""
neuroview — Demo Usage  

Dependencies:
    pip install plotly pandas numpy scipy scikit-image kaleido
"""

In [ ]:
from pathlib import Path
import numpy as np 
import pandas as pd
import os
import sys

module_path = os.path.abspath('/Volumes/ZEKAI iEEG')
if module_path not in sys.path:
    sys.path.append(module_path)

import neuroview as nv
from neuroview import BrainScene, Surface, Volume, ElectrodeSet

# CONFIGURE YOUR PATHS HERE 

PIAL_LH  = "fsaverage/surf/lh.pial"
PIAL_RH  = "fsaverage/surf/rh.pial"
GIFTI_LH = "fsaverage/surf/lh.pial.gii"
MRI_NII  = "MNI_T1_ac.nii"
SEG_NII  = "Segmentations/cingulate.nii"
ELEC_TSV = ("/sub-ccepAgeUMCU36/ses-1/ieeg/sub-ccepAgeUMCU36_ses-1_electrodes.tsv")

In [ ]:
# EXAMPLE 1 — Simple glass brain + electrodes 

def example_glass_brain():
    """Glass brain: transparent pial surface + electrode scatter.""" 

    brain    = Surface.from_file(PIAL_LH, name="LH pial", color="#B8C4D6", opacity=0.5)
    elec     = ElectrodeSet.from_file(ELEC_TSV)
    elec_lh  = elec.left_hemisphere()

    print(brain)
    print(elec_lh)

    scene = (
        BrainScene(width=1100, height=850)
        .set_camera(azimuth=-90, elevation=0)
        .add_glass_brain(brain)
        .add_electrodes(elec_lh, color="gold", size=7, show_labels=True)
        .set_title("Glass Brain — fsaverage LH")
    )
    scene.show() 

example_glass_brain()

In [ ]:
# EXAMPLE 2 — Group-coloured electrodes (by anatomical label column) 

def example_grouped_electrodes():
    """Electrodes colour-coded by Destrieux parcellation label.""" 

    brain    = Surface.from_file(PIAL_LH, opacity=0.18)
    elec     = ElectrodeSet.from_file(ELEC_TSV).left_hemisphere()
    GROUP_COL = "Destrieux_label_text"   # ← adjust to your column name

    if GROUP_COL not in elec.df.columns:
        print(f"  Column '{GROUP_COL}' not found. Available: {list(elec.df.columns)}")
        return

    rec_coord = elec.get_electrode("Fp01")   # ← adjust channel name

    scene = (
        BrainScene(width=1200, height=900)
        .set_camera(azimuth=-90, elevation=10)
        .add_glass_brain(brain)
        .add_electrodes_grouped(elec, GROUP_COL, cmap="Turbo", size=7)
        .set_title("Grouped Electrodes — fsaverage LH")
    )
    if rec_coord is not None:
        scene.add_highlight_electrode(rec_coord, label="Fp01", color="red", size=14)

    scene.show() 

example_grouped_electrodes()

In [ ]:
# EXAMPLE 3 — Scalar-weighted electrodes (CCEP amplitude) 

def example_weighted_electrodes():
    """Electrodes scaled and coloured by a scalar weight (e.g. CCEP amplitude).""" 

    brain  = Surface.from_file(PIAL_LH, opacity=0.18)
    elec   = ElectrodeSet.from_file(ELEC_TSV).left_hemisphere()
    rng    = np.random.default_rng(42)
    weights = rng.normal(loc=0, scale=1.5, size=elec.n)  # synthetic weights for demo; replace with your data column
    scene = (
        BrainScene(width=1100, height=850)
        .set_camera(azimuth=-90, elevation=0)
        .add_glass_brain(brain)
        .add_electrodes_weighted(
            elec, weights=weights, colorscale="RdBu_r",
            size_range=(5, 16), name="CCEP amplitude",
        )
        .set_title("Weighted Electrodes")
    )
    scene.show() 

example_weighted_electrodes()

In [ ]:
# EXAMPLE 4 — Glass brain + MRI slice planes in 3-D 

def example_mri_slices():
    """Transparent glass brain with axial + coronal MRI slice planes.""" 

    brain = Surface.from_file(PIAL_LH, opacity=1)
    vol   = Volume.from_file(MRI_NII)
    elec  = ElectrodeSet.from_file(ELEC_TSV).left_hemisphere()
    print(vol)

    scene = (
        BrainScene(width=1200, height=900)
        .set_camera(azimuth=-90, elevation=20)
        .add_glass_brain(brain)
        .add_mri_slice(vol, plane="axial",   coord_mm=15.0,   opacity=1) 
        .add_electrodes(elec, color="gold", size=7)
        .set_title("Glass Brain + MRI Slices")
    )
    scene.show()

example_mri_slices()

In [ ]:
# EXAMPLE 5 — Glass brain + segmented anatomical structure

def example_segmentation():
    """Glass brain + segmented anatomical structure.""" 

    brain = Surface.from_file(PIAL_LH, opacity=.2)
    seg   = Volume.from_file(SEG_NII)

    scene = (
        BrainScene(width=1100, height=850)
        .set_camera(azimuth=-90, elevation=0)
        .add_glass_brain(brain)
        .add_segmentation(seg, label=1, smooth_sigma=0.8,
                          color="#EDB085", opacity=0.75, name="Cingulate")
        .set_title("Glass Brain + Cingulate")
    )
    scene.show() 
example_segmentation()

In [ ]:
# EXAMPLE 6 — Full scene: Glass brain + MRI slice + electrodes

def example_full_scene():
    """Glass brain, MRI slice, electrodes.""" 

    brain = Surface.from_file(PIAL_LH, opacity=1)
    elec  = ElectrodeSet.from_file(ELEC_TSV).left_hemisphere()
    vol   = Volume.from_file(MRI_NII)
    GROUP_COL = "Destrieux_label_text"
    rec_coord = elec.get_electrode("Fp01")

    scene = (
        BrainScene(width=1300, height=950)
        .set_camera(azimuth=-90, elevation=15)
        .add_glass_brain(brain)
        .add_mri_slice(vol, "coronal", coord_mm=-15.0, opacity=1)
    )
    if GROUP_COL in elec.df.columns:
        scene.add_electrodes_grouped(elec, GROUP_COL, cmap="Turbo")
    else:
        scene.add_electrodes(elec, color="gold")
    if rec_coord is not None:
        scene.add_highlight_electrode(rec_coord, label="Fp01", color="red")

    scene.set_title("Full Scene — Glass Brain + MRI + Electrodes")
    scene.show()

example_full_scene()

In [ ]:
# EXAMPLE 7 — Bilateral glass brain with different opacity/colour per hemisphere

def example_bilateral():
    """Both hemispheres with different opacity/colour.""" 

    lh   = Surface.from_file(PIAL_LH, name="LH", color="#B8C4D6", opacity=0.18)
    rh   = Surface.from_file(PIAL_RH, name="RH", color="#C4B8D6", opacity=0.18)
    elec = ElectrodeSet.from_file(ELEC_TSV)

    scene = (
        BrainScene(width=1200, height=900)
        .set_camera(azimuth=0, elevation=20)
        .add_glass_brain(lh)
        .add_glass_brain(rh)
        .add_electrodes(elec.left_hemisphere(),  color="gold",    size=7, name="LH electrodes")
        .add_electrodes(elec.right_hemisphere(), color="skyblue", size=7, name="RH electrodes")
        .set_title("Bilateral ECoG")
    )
    scene.show()

example_bilateral()

In [ ]:
def example_surface_activity():
    """Gaussian projection of electrode weights onto the cortical surface."""

    brain   = Surface.from_file(PIAL_LH, opacity=0.18)
    elec    = ElectrodeSet.from_file(ELEC_TSV).left_hemisphere()
    
    rng     = np.random.default_rng(0)
    weights = rng.normal(loc=0, scale=1.5, size=elec.n)  # synthetic z-scores 

    scene = (
        BrainScene(width=1100, height=850)
        .set_camera(azimuth=-90, elevation=0) 
        .add_surface_activity(
            surface          = brain,
            electrode_coords = elec.coords,
            weights          = weights,
            sigma            = 50,
            colorscale       = "RdBu_r", 
            symmetric_clim   = True,    
            opacity          = 1.0,
            colorbar_title   = "z-score",
            name             = "Surface activity",
        )
        
        .add_electrodes_weighted(
            elec, weights=weights, colorscale="RdBu_r",
            size_range=(5, 14), show_labels=True, name="Electrodes",
        )
        .set_title("Gaussian Electrode-Cortex Projection")
    )
    scene.show()

example_surface_activity()

In [ ]:
# EXAMPLE 9 — Glass brain + MRI slice planes + deep (thalamic) electrode
#             + Gaussian surface activity projection


def example_thalamic_electrode_with_activity():
    """Glass brain + 3 MRI slice planes + thalamic DBS electrode + surface activity."""
    
    lh  = Surface.from_file(PIAL_LH, name="LH pial", color="#B8C4D6", opacity=0.15)
    rh  = Surface.from_file(PIAL_RH, name="RH pial", color="#C4B8D6", opacity=0.15)
    vol = Volume.from_file(MRI_NII)
    print(vol)
    
    ctx_elec = ElectrodeSet.from_file(ELEC_TSV)
    ctx_lh   = ctx_elec.left_hemisphere()
    
    thal_coords = np.array([
        [11.0, -19.0,  9.0],   # central contact (VPL / Vim centroid)
        [ 9.5, -18.5, 12.0],   # dorsal contact
        [12.5, -19.5,  6.0],   # ventral contact
    ])
    thal_names = ["Thal_mid", "Thal_dors", "Thal_vent"]
    thal_elec  = ElectrodeSet.from_arrays(thal_coords, thal_names, name="Thalamus")

    # Synthetic CCEP weights on cortical electrodes
    
    rng     = np.random.default_rng(7)
    weights = rng.normal(0, 1.0, ctx_lh.n)
    weights[:3]  += 3.0   # simulate strong frontal CCEP
    
    scene = BrainScene(width=1300, height=950, bg_color="white")
    
    scene.set_camera(azimuth=-70, elevation=15, distance=2.6)
    
    scene.add_glass_brain(lh)
    scene.add_glass_brain(rh)
    
    scene.add_mri_slice(vol, plane="axial",    coord_mm=8.0,   opacity=0.90,
                        colorscale="Gray", name="Axial z=8 mm")
    
    scene.add_surface_activity(
        surface          = lh,
        electrode_coords = ctx_lh.coords,
        weights          = weights,
        sigma            = 50,           # mm^2
        colorscale       = "RdBu_r",
        symmetric_clim   = True,
        opacity          = 1,
        colorbar_title   = "CCEP (z)",
        name             = "Cortical CCEP",
    )
    
    scene.add_electrodes_weighted(
        ctx_lh, weights=weights, colorscale="RdBu_r",
        size_range=(5, 13), show_labels=False, name="ECoG electrodes",
    )
    
    scene.add_electrodes(
        thal_elec, color="yellow", size=5,
        show_labels=False, label_size=10, name="Thalamus",
    )

    scene.set_title(
        "Glass Brain + MRI Slices + Thalamic Electrode + Cortical CCEP"
    )
    scene.show()

example_thalamic_electrode_with_activity()